In [1]:
!pip install -q transformers datasets evaluate accelerate pynvml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.1 MB/s eta 0:00:00


In [2]:
import time, gc, warnings, threading
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset
import evaluate as ev

warnings.filterwarnings('ignore')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

MODELS = {
    'TinyBERT':   'huawei-noah/TinyBERT_General_4L_312D',
    'DistilBERT': 'distilbert-base-uncased',
    'AlBERT':     'albert-base-v2',
    'MobileBERT': 'google/mobilebert-uncased',
    'BERT-base':  'bert-base-uncased',
}

DATASETS = {
    'SST2': ('stanfordnlp/sst2',  None,   'validation',         'sentence',               'label'),
    'QNLI': ('nyu-mll/glue',      'qnli', 'validation',         ('question','sentence'),   'label'),
    'MNLI': ('nyu-mll/glue',      'mnli', 'validation_matched', ('premise','hypothesis'),  'label'),
    'QQP':  ('nyu-mll/glue',      'qqp',  'validation',         ('question1','question2'), 'label'),
    'RTE':  ('nyu-mll/glue',      'rte',  'validation',         ('sentence1','sentence2'), 'label'),
}

BATCH_SIZE      = 32
MAX_SAMPLES     = 500
MAX_LENGTH      = 128
QUANT_BITS      = 8
POLL_INTERVAL_S = 0.005  # 5 ms polling interval for power sampler


Device: cuda


In [3]:
# ---------------------------------------------------------------------------
# pynvml calls the NVML C library directly (no process spawn overhead),
# allowing us to poll at ~5 ms intervals and time-average power across the
# entire forward pass instead of taking a single snapshot before it.
# ---------------------------------------------------------------------------
try:
    import pynvml
    pynvml.nvmlInit()
    _nvml_handle = pynvml.nvmlDeviceGetHandleByIndex(0)
    NVML_AVAILABLE = True
    print(f'pynvml ready | GPU: {pynvml.nvmlDeviceGetName(_nvml_handle)}')
except Exception as e:
    NVML_AVAILABLE = False
    print(f'pynvml unavailable ({e}) — energy will be reported as 0.0 mJ')


class PowerSampler:
    def __init__(self):
        self._samples = []
        self._running = False
        self._thread  = None

    def _poll(self):
        while self._running:
            if NVML_AVAILABLE:
                try:
                    # nvmlDeviceGetPowerUsage returns milliwatts directly (no conversion needed)
                    self._samples.append(pynvml.nvmlDeviceGetPowerUsage(_nvml_handle))
                except Exception:
                    pass
            time.sleep(POLL_INTERVAL_S)

    def start(self):
        self._samples = []
        self._running = True
        self._thread  = threading.Thread(target=self._poll, daemon=True)
        self._thread.start()

    def stop(self):
        """Stop sampling and return mean power in mW (0.0 if unavailable)."""
        self._running = False
        self._thread.join(timeout=0.5)
        return float(np.mean(self._samples)) if self._samples else 0.0


pynvml ready | GPU: Tesla T4


In [4]:
def get_model(hf_id, num_labels):
    tok = AutoTokenizer.from_pretrained(hf_id)
    model = AutoModelForSequenceClassification.from_pretrained(
        hf_id, num_labels=num_labels, ignore_mismatched_sizes=True
    ).half().to(DEVICE)
    return tok, model


def memory_mb(model, quant_bits=8):
    # Each FP16 parameter = 2 bytes.
    # INT8 uses 1 byte per parameter (quant_bits/16 = 8/16 = 0.5).
    fp16_bytes = sum(p.numel() * 2 for p in model.parameters())
    return round(fp16_bytes * (quant_bits / 16) / 1e6, 2)


def tokenize(tok, texts):
    if isinstance(texts[0], tuple):
        return tok([t[0] for t in texts], [t[1] for t in texts],
                   truncation=True, padding=True, max_length=MAX_LENGTH, return_tensors='pt')
    return tok(texts, truncation=True, padding=True,
               max_length=MAX_LENGTH, return_tensors='pt')


def benchmark(model, tok, ds_cfg):
    path, config, split, text_col, label_col = ds_cfg
    ds = load_dataset(path, config, split=split) if config else load_dataset(path, split=split)
    ds = ds.select(range(min(MAX_SAMPLES, len(ds))))

    model.eval()
    preds, labels, latencies, energies = [], [], [], []
    t_start = time.perf_counter()

    for i in range(0, len(ds), BATCH_SIZE):
        batch = ds[i:i+BATCH_SIZE]
        texts = list(zip(batch[text_col[0]], batch[text_col[1]])) if isinstance(text_col, tuple) else batch[text_col]
        enc = {k: v.to(DEVICE) for k, v in tokenize(tok, texts).items()}

        sampler = PowerSampler()
        sampler.start()

        t0 = time.perf_counter()
        with torch.no_grad():
            out = model(**enc)

        if DEVICE == 'cuda':
            torch.cuda.synchronize()
        elapsed_ms = (time.perf_counter() - t0) * 1000

        avg_power_mw = sampler.stop()

        batch_size_actual = len(batch[label_col])
        energy_mj = (avg_power_mw * elapsed_ms * 1e-3) / batch_size_actual

        latencies.append(elapsed_ms / batch_size_actual)
        energies.append(energy_mj)
        preds.extend(out.logits.argmax(-1).cpu().tolist())
        labels.extend(batch[label_col])

    total_time = time.perf_counter() - t_start
    acc        = ev.load('accuracy').compute(predictions=preds, references=labels)['accuracy']
    lat_ms     = np.mean(latencies)
    throughput = len(ds) / total_time
    energy_mj  = np.mean(energies)
    return round(acc * 100, 2), round(lat_ms, 4), round(throughput, 1), round(energy_mj, 4)


In [5]:
NUM_LABELS = {'SST2': 2, 'QNLI': 2, 'MNLI': 3, 'QQP': 2, 'RTE': 2}
results = []

for ds_name, ds_cfg in DATASETS.items():
    for model_name, hf_id in MODELS.items():
        print(f'{ds_name} | {model_name} ... ', end='', flush=True)
        try:
            tok, model = get_model(hf_id, NUM_LABELS[ds_name])
            mem = memory_mb(model, QUANT_BITS)
            acc, lat, tp, energy = benchmark(model, tok, ds_cfg)
            results.append({
                'Dataset': ds_name, 'Model': model_name,
                'Memory (MB)': mem, 'Latency (ms)': lat,
                'Accuracy (%)': acc, 'Bits': QUANT_BITS,
                'Energy (mJ)': energy, 'Throughput (sps)': tp
            })
            print(f'Acc={acc:.1f}%  Lat={lat:.3f}ms  Mem={mem}MB  Energy={energy:.4f}mJ  TP={tp:.0f}sps')
        except Exception as e:
            print(f'ERROR: {e}')
        finally:
            try:
                del model, tok
                gc.collect()
                torch.cuda.empty_cache()
            except:
                pass


SST2 | TinyBERT ... 

config.json:   0%|          | 0.00/409 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/62.7M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

...

MobileBertForSequenceClassification LOAD REPORT from: google/mobilebert-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
mobilebert.embeddings.position_ids         | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.dense.weight               | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignore

Acc=52.7%  Lat=3.751ms  Mem=24.58MB  Energy=125.0541mJ  TP=199sps
RTE | BERT-base ... 

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Acc=46.9%  Lat=1.457ms  Mem=109.48MB  Energy=64.8084mJ  TP=388sps


In [6]:
df = pd.DataFrame(results).set_index(['Dataset', 'Model'])
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 150)
pd.set_option('display.float_format', '{:.4f}'.format)

for ds in df.index.get_level_values('Dataset').unique():
    print(f'\n========== {ds} ==========')
    print(df.loc[ds].to_string())
print()



========== SST2 ==========
            Memory (MB)  Latency (ms)  Accuracy (%)  Bits  Energy (mJ)  Throughput (sps)
Model                                                                                   
TinyBERT        14.3500        1.8713       49.6000     8      52.2726          456.0000
DistilBERT      66.9600        0.4508       51.6000     8      16.3018         1211.7000
AlBERT          11.6900        1.0526       47.0000     8      48.5127          675.4000
MobileBERT      24.5800        1.2582       47.0000     8      40.5220          661.6000
BERT-base      109.4800        0.5949       46.8000     8      26.6883         1126.7000

========== QNLI ==========
            Memory (MB)  Latency (ms)  Accuracy (%)  Bits  Energy (mJ)  Throughput (sps)
Model                                                                                   
TinyBERT        14.3500        0.4406       52.4000     8      16.3073          857.9000
DistilBERT      66.9600        0.6717       47.4000   